# HMK / 03 — Are the daily, 1 hour and 15 minute state probabilities independent? (descriptive)

Protocol: [`HMK 006`](../../discussions/MNQ_Hierarchical_Markov/006_claude_hmk03_tier_independence.md). Code: `src/equity_trading/hmk.py`.

- **States** as in 02: daily 20 days ±5%; 1 h W = 40 / 60; 15 m W = 60 / 80; each intraday tier by (a) the set √time band or (b) the 3-state HMM on the window return (fitted once on the window, then a fixed classifier). 2 methods × 2 × 2 windows = 8 cases.
- **Parent of a bar** = the parent's last completed bar (15 m in day d → daily state of d−1; 15 m in hour h → 1 h state of h−1; 1 h in day d → daily state of d−1).
- **Links:** 15m | 1d, 1h | 1d, 15m | 1h.
- **Two questions per link:**
  1. Is the child's state independent of the parent's? P(child | parent) = P(child) — mutual information I(child; parent).
  2. Is the child's *transition* independent of the parent? P(child_{t+1} | child_t, parent) = P(child_{t+1} | child_t) — conditional mutual information I(child_{t+1}; parent | child_t). This is the (P of 15 m state at t+1 | current 1D state) question.
- **Two references:** (i) *circular shift* of the parent's states by ≥ 60 whole days (keeps each tier's own persistence): tests independence as such. (ii) *shuffled returns*: 15 m returns permuted within each time-of-day slot, all tiers rebuilt from them, same classifiers: the dependence that shared returns (the parent's window contains the child's bars) produce on their own.
- **Window:** 2020-01 – 2023-06 (halves reported without references).

Environment: `HMK_SYNTHETIC=1` (random-walk smoke run), `HMK_SYNTH_DIR`.

In [ ]:
import json, os, sys, tempfile, warnings, itertools, time
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
SYNTHETIC = os.environ.get("HMK_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("HMK_SYNTH_DIR") or tempfile.mkdtemp(prefix="hmk_")
    os.environ["HMK_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import hmk as H
from equity_trading import micro_bars as MB

D = dt.date
CFG = H.HMKConfig()
if SYNTHETIC:
    WINDOWS = {"all": (D(2020, 1, 1), D(2021, 6, 30)), "H1": (D(2020, 1, 1), D(2020, 9, 30)), "H2": (D(2020, 10, 1), D(2021, 6, 30))}
    if not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
        H.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.load_start, WINDOWS["all"][1], effect=0.0)
    N_SHIFT, N_SHUFFLE = 50, 20
else:
    WINDOWS = {"all": (D(2020, 1, 1), D(2023, 6, 30)), "H1": (D(2020, 1, 1), D(2021, 9, 30)), "H2": (D(2021, 10, 1), D(2023, 6, 30))}
    N_SHIFT, N_SHUFFLE = 500, 200
OUT = Path("03"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50)
LINKS = ("15m|1d", "1h|1d", "15m|1h")
print("engine", H.HMK_VERSION, "| synthetic random walk" if SYNTHETIC else "| real data", "| references", N_SHIFT, N_SHUFFLE)

In [ ]:
m1 = MB.load_1m(CFG.root, CFG.load_start, WINDOWS["all"][1] + dt.timedelta(days=1))
BARS = H.session_bars(m1, CFG)
del m1
end = WINDOWS["all"][1]
BARS = {k: H.in_window(v, D(1900, 1, 1), end).reset_index(drop=True) for k, v in BARS.items()}
T = H.tiers_from_15m(BARS)
R15 = T["q"].ret_bp.to_numpy(float)
SLOT = T["q"].slot.to_numpy()
QD = pd.to_datetime(T["q"].trade_date).dt.date.to_numpy()
def masks_for(a, b):
    return {"15m": (QD >= a) & (QD <= b), "1h": np.array([(d >= a) & (d <= b) for d in T["hour_dates"]]),
            "1d": np.array([(d >= a) & (d <= b) for d in T["day_dates"]])}
M_ALL = masks_for(*WINDOWS["all"])
R = H.tier_returns_from(R15, T)
# HMMs on the window return, fitted once on the full window, then used as fixed classifiers
HMM = {}
for tier, w in (("1h", 40), ("1h", 60), ("15m", 60), ("15m", 80)):
    _, _, HMM[(tier, w)] = H.hmm_states(R[tier], w, M_ALL[tier], CFG)
    print(tier, w, "HMM means %", (HMM[(tier, w)].mu / 100).round(3))
CASES = {}
for method, w1, w15 in itertools.product(("band", "hmm"), (40, 60), (60, 80)):
    def spec(tier, w):
        return (H.TierSpec(tier, w, "band", H.band_for_window(tier, w)) if method == "band"
                else H.TierSpec(tier, w, "hmm", model=HMM[(tier, w)]))
    CASES[f"{method}_1hW{w1}_15mW{w15}"] = {"1d": H.TierSpec("1d", 20, "band", 0.05), "1h": spec("1h", w1), "15m": spec("15m", w15)}
print(list(CASES))

## 1. Statistics and references

In [ ]:
rng = np.random.default_rng(0)
STATS, REAL = [], {}
T0 = time.time()
SHUF_R = [H.shuffle_within_slot(R15, SLOT, rng) for _ in range(N_SHUFFLE)]
for name, specs in CASES.items():
    real = H.dependence(R15, T, specs, M_ALL); REAL[name] = real
    shift = H.circular_shift_null(R15, T, specs, M_ALL, N_SHIFT, rng)
    shuf = [H.dependence(r, T, specs, M_ALL) for r in SHUF_R]
    for link in LINKS:
        row = {"case": name, "link": link, "n": real[link]["n"]}
        for stat in ("mi", "cmi"):
            x = real[link][stat]
            s1 = np.array(shift[link][stat]); s2 = np.array([s[link][stat] for s in shuf])
            row.update({stat: x, f"{stat}_shift_mean": s1.mean(), f"{stat}_shift_q95": np.quantile(s1, .95),
                        f"{stat}_p_shift": (1 + (s1 >= x).sum()) / (1 + len(s1)),
                        f"{stat}_shuffle_mean": s2.mean(), f"{stat}_shuffle_q95": np.quantile(s2, .95),
                        f"{stat}_p_shuffle": (1 + (s2 >= x).sum()) / (1 + len(s2)), f"{stat}_excess_vs_shuffle": x - s2.mean()})
        STATS.append(row)
    print(name, f"{time.time() - T0:.0f}s", flush=True)
STATS = pd.DataFrame(STATS); STATS.to_csv(OUT / "03_stats.csv", index=False)
show = ["case", "link", "n", "mi", "mi_shift_q95", "mi_p_shift", "mi_shuffle_mean", "mi_p_shuffle",
        "cmi", "cmi_shift_q95", "cmi_p_shift", "cmi_shuffle_mean", "cmi_p_shuffle"]
display(STATS[show].round(5))

## 2. P(child state | parent state)

Rows: the parent's last completed state. A row equal to the child's overall shares means independence.

In [ ]:
TAB = []
for name, real in REAL.items():
    for link in LINKS:
        C = real[link]["counts"]
        P = H.cond_tables(C); P.loc["overall"] = C.sum(1) / C.sum()
        TAB.append(P.reset_index(names="row").assign(case=name, link=link))
TAB = pd.concat(TAB); TAB.to_csv(OUT / "03_conditional_states.csv", index=False)
for name in CASES:
    print("===", name)
    display(TAB[TAB.case == name].set_index(["link", "row"])[list(H.STATE_NAMES)].round(3))

## 3. P(child_{t+1} | child_t, parent) — the child's transition matrix under each parent state

If the three parent rows match 'pooled', the parent does not change the child's transitions.

In [ ]:
TR = []
for name, real in REAL.items():
    for link in LINKS:
        for g, M in H.cond_transitions(real[link]["counts3"]).items():
            TR.append(M.reset_index(names="from").assign(case=name, link=link, parent=g,
                      n=int(real[link]["counts3"].sum() if g == "pooled" else real[link]["counts3"][list(H.STATE_NAMES).index(g.split()[-1])].sum())))
TR = pd.concat(TR); TR.to_csv(OUT / "03_conditional_transitions.csv", index=False)
for name in CASES:
    print("===", name)
    display(TR[(TR.case == name)].pivot_table(index=["link", "parent", "from"], values=list(H.STATE_NAMES), sort=False)[list(H.STATE_NAMES)].round(3))

## 4. Halves (no references)

In [ ]:
HALF = []
for win in ("H1", "H2"):
    mk = masks_for(*WINDOWS[win])
    for name, specs in CASES.items():
        d = H.dependence(R15, T, specs, mk)
        for link in LINKS:
            HALF.append({"window": win, "case": name, "link": link, "mi": d[link]["mi"], "cmi": d[link]["cmi"], "n": d[link]["n"]})
HALF = pd.DataFrame(HALF)
both = pd.concat([HALF, STATS[["case", "link", "mi", "cmi", "n"]].assign(window="all")])
both.to_csv(OUT / "03_halves.csv", index=False)
display(both.pivot_table(index=["case", "link"], columns="window", values=["mi", "cmi"], sort=False).round(5))

## 5. Heatmaps: 15 m transitions under each daily state (all cases)

In [ ]:
fig, ax = plt.subplots(len(CASES), 4, figsize=(13, 3.0 * len(CASES)))
for i, (name, real) in enumerate(REAL.items()):
    for j, (g, M) in enumerate(H.cond_transitions(real["15m|1d"]["counts3"]).items()):
        a = ax[i, j]; v = M.to_numpy(float); a.imshow(v, vmin=0, vmax=1, cmap="Blues")
        for (p, q), x in np.ndenumerate(v):
            a.text(q, p, f"{x:.2f}" if np.isfinite(x) else "-", ha="center", va="center", color="k" if not np.isfinite(x) or x < 0.6 else "w", fontsize=8)
        a.set_xticks(range(3), H.STATE_NAMES, fontsize=7); a.set_yticks(range(3), H.STATE_NAMES, fontsize=7)
        a.set_title(f"{name}\n15m | daily {g}", fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "03_transitions_15m_by_daily.png", dpi=100); plt.show()
json.dump({"version": H.HMK_VERSION, "synthetic": SYNTHETIC, "windows": {k: [str(a), str(b)] for k, (a, b) in WINDOWS.items()},
           "n_shift": N_SHIFT, "n_shuffle": N_SHUFFLE, "cases": list(CASES),
           "hmm_means_pct": {f"{t}_W{w}": (m.mu / 100).round(4).tolist() for (t, w), m in HMM.items()}},
          open(OUT / "03_run.json", "w"), indent=1)